# Qwen3.5-4B: asignar prioridades a un conjunto de tareas

Cada entrada contiene **cuatro tareas**. El LLM asigna a cada una **Critical,
High, Medium o Low**, junto con una razón breve. Conserva cada ID una vez.
La evaluación ignora el orden de salida; no construimos horarios.

Comparamos Qwen original, LoRA y DoRA con PEFT. Todo el código y los experimentos
están aquí, conservando el flujo del docente. Deadline, modo predeterminado para
la entrega en cuatro horas, usa 80/12/12 conjuntos y una época por adaptador.
Se omiten factorial, réplicas, datos manuales, regresión, costos monetarios y PPT.
No se presupone una mejora tras adaptar.


## 1. Preparar Khipu y el entorno

Seguir `AGENTS.md`: el login admite instalación y transferencia; **no cargar el
modelo ni entrenar allí**. El experimento requiere un job GPU de Slurm.
No introducir tokens de Hugging Face en las celdas. Este modelo público no exige login.

`requerements.txt` es el archivo original: incluye `pywinpty` (Windows), un Torch
CUDA específico y no incluye Matplotlib ni scikit-learn. El nuevo `requirements.txt`
conserva las dependencias necesarias y evita trasladar ese entorno completo.

Preparación desde esta carpeta (Python 3.11):

```bash
# Este Python está en almacenamiento compartido; /usr/bin/python3.11 no existe en ds001.
KHIPU_PYTHON=/opt/ohpc/pub/apps/spack/local/linux-rocky8-zen2/aocc-4.2.0/python-3.11.7-ujz6vknvxczz5byyrbuykxbs7o4obmpw/bin/python3.11
"$KHIPU_PYTHON" -m venv .venv
.venv/bin/python -m pip install --upgrade pip
# CUDA 12.6: verificar compatibilidad con el driver del nodo GPU.
.venv/bin/python -m pip install torch==2.9.1 --index-url https://download.pytorch.org/whl/cu126
.venv/bin/python -m pip install -r requirements.txt
# Ejecuta solo la celda de descarga; no carga pesos en RAM/GPU.
.venv/bin/python - <<'PY'
import nbformat
n = nbformat.read('qwen35_4b_priorizacion.ipynb', as_version=4)
for c in n.cells:
    if 'download' in c.metadata.get('tags', []):
        exec(c.source)
PY
```

Después, `sbatch --time=00:30:00 run_qwen.sbatch smoke`; revisar el resultado antes de ejecutar
`sbatch --time=02:00:00 run_qwen.sbatch deadline`. El wrapper solicita siete shards RTX A6000 de g002, 4 CPUs y 48 GiB
de RAM. Los shards no reservan una cantidad aislada de VRAM; se comprueba memoria libre. Smoke limita la ejecución a 30 minutos; full dispone de 4 horas. Revisar
disponibilidad y límites con `sinfo`, `sacctmgr` y `squeue` antes de enviar.
No se supone que una fracción GPU reserve una cantidad aislada de VRAM.

Para una sesión Jupyter interactiva, abrir este notebook con el kernel del entorno
**dentro de una asignación GPU** y ejecutar de arriba hacia abajo.


## 2. Descargar el checkpoint oficial (sin cargar el modelo)

Esta celda es autocontenida y se puede ejecutar por separado en el login.
La primera descarga resuelve `main` a un commit y lo registra en `models/model_manifest.json`.
Las siguientes reutilizan ese commit; no siguen automáticamente actualizaciones.
Se descargan los archivos del repositorio completo, incluidos pesos y tokenizer.


In [1]:
import json
import os
from pathlib import Path

if os.environ.get("QWEN_LOCAL_ONLY") != "1":
    # Standard HTTPS avoids the stalled Xet transport observed on this cluster.
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("HF_HOME", str(Path("models/hf-home").resolve()))
    from huggingface_hub import HfApi, snapshot_download

    model_id = "Qwen/Qwen3.5-4B"
    model_root = Path("models").resolve()
    model_root.mkdir(exist_ok=True)
    manifest_path = model_root / "model_manifest.json"
    previous = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    if previous and previous["model_id"] != model_id:
        raise ValueError("El manifiesto existente pertenece a otro checkpoint.")
    revision = previous["revision"] if previous else HfApi().model_info(model_id).sha
    snapshot = snapshot_download(
        repo_id=model_id, revision=revision, cache_dir=str(model_root / "hub"),
        max_workers=4,
    )
    manifest = {"model_id": model_id, "revision": revision,
                "snapshot_path": str(Path(snapshot).resolve())}
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    print("Checkpoint disponible:", manifest)
else:
    print("Modo local: usar el checkpoint preparado antes de Slurm.")


Modo local: usar el checkpoint preparado antes de Slurm.


## 3. Configuración y trazabilidad

Deadline: 80 conjuntos de entrenamiento, 12 de validación y 12 de prueba; cuatro
tareas por conjunto, una época y hasta 224 tokens generados. Smoke usa 8/4/4,
dos pasos y acumulación 1. Full opcional conserva 400/100/100 y tres épocas.
Prepare verifica datos sin cargar un LLM. Semilla 42, mismos datos y presupuesto
entre métodos, con revisión del checkpoint y versiones guardadas.


In [2]:
import gc
import hashlib
import importlib.metadata as metadata
import math
import platform
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, f1_score, recall_score

@dataclass
class Config:
    model_id: str = "Qwen/Qwen3.5-4B"
    seed: int = 42
    mode: str = os.environ.get("QWEN_MODE", "deadline")
    train_size: int = 400
    validation_size: int = 100
    test_size: int = 100
    rank: int = 16
    alpha: int = 32
    dropout: float = 0.0
    learning_rate: float = 2e-4
    epochs: int = 3
    batch_size: int = 1
    accumulation: int = 16
    max_seq_length: int = 768
    max_new_tokens: int = 224
    warmup_ratio: float = 0.03

config = Config()
if config.mode not in {"full", "smoke", "prepare", "deadline"}:
    raise ValueError("QWEN_MODE debe ser full, smoke, prepare o deadline.")
if config.mode == "smoke":
    config.train_size, config.validation_size, config.test_size = 8, 4, 4
    config.accumulation = 1
if config.mode == "deadline":
    config.train_size, config.validation_size, config.test_size = 80, 12, 12
    config.epochs = 1
    config.max_new_tokens = 224
run_id = os.environ.get("QWEN_RUN_ID") or (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + config.mode
)
if Path(run_id).name != run_id:
    raise ValueError("QWEN_RUN_ID debe ser un nombre de carpeta, sin rutas.")
run_dir = Path("outputs") / run_id
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "figures").mkdir()
random.seed(config.seed)
np.random.seed(config.seed)

def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False) + "\n", encoding="utf-8")

save_json(run_dir / "config.json", asdict(config))
packages = {p.metadata["Name"]: p.version for p in metadata.distributions()}
environment = {"python": sys.version, "platform": platform.platform(),
               "packages": packages, "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
               "status": "prepared", "mode": config.mode}
save_json(run_dir / "environment.json", environment)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        check=True, capture_output=True, text=True)
(run_dir / "requirements-frozen.txt").write_text(freeze.stdout)
plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120, "savefig.dpi": 160})
COLORS = {"Original": "#326A9F", "LoRA": "#A46E13", "DoRA": "#98658B"}
TASKS_PER_SET = 4
PRIORITIES = ["Critical", "High", "Medium", "Low"]

def show_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(run_dir / "figures" / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Resultados:", run_dir.resolve())
display(pd.DataFrame([asdict(config)]).T.rename(columns={0: "valor"}))


Resultados: /home/jeffrey.monja/Github/2026-II/PlanIA/S10/Lora Scripts/outputs/58321_deadline


,valor
model_id,Qwen/Qwen3.5-4B
seed,42
mode,deadline
train_size,80
validation_size,12
test_size,12
rank,16
alpha,32
dropout,0.0
learning_rate,0.0002


## 4. Política de las cuatro prioridades

Aplicar la primera regla satisfecha por cada tarea:

- **Critical:** incidente activo, o bloqueo de impacto 5 sin alternativa.
- **High:** bloqueo sin alternativa, o plazo de hasta 24 horas con impacto ≥3.
- **Medium:** alternativa disponible, plazo de hasta siete días, o impacto ≥3.
- **Low:** demás casos; mantenimiento sin urgencia ni impacto significativo.

Los campos explícitos controlan la prioridad; la insistencia de una descripción
no la altera. Se devuelve JSON `tasks` con id, priority y reason para cada entrada.
Una razón breve es obligatoria, pero su calidad se revisa cualitativamente.


In [3]:
POLICY = (
    "Asigna una prioridad a CADA tarea del conjunto. Usa exactamente Critical, High, Medium o Low. "
    "Aplica la primera regla satisfecha: Critical si active_incident=true, o si blocked=true, "
    "impact=5 y workaround=false. High si blocked=true y workaround=false, o si "
    "deadline_hours<=24 e impact>=3. Medium si workaround=true, deadline_hours<=168 "
    "o impact>=3. Low en los demás casos. impact va de 1 (menor) a 5 (crítico). "
    "La insistencia de la descripción no cambia los criterios explícitos. "
    "Incluye cada id exactamente una vez, con una razón breve de hasta 12 palabras. "
    'Devuelve solo JSON: {"tasks":[{"id":"T1","priority":"High","reason":"..."},...]}. '
    "Conserva los IDs recibidos."
)

def priority_for(t):
    if t["active_incident"] or (t["blocked"] and t["impact"] == 5 and not t["workaround"]):
        return "Critical"
    if (t["blocked"] and not t["workaround"]) or (t["deadline_hours"] <= 24 and t["impact"] >= 3):
        return "High"
    if t["workaround"] or t["deadline_hours"] <= 168 or t["impact"] >= 3:
        return "Medium"
    return "Low"

def reason_for(t):
    priority = priority_for(t)
    if priority == "Critical":
        return "Incidente activo." if t["active_incident"] else "Bloqueo crítico sin alternativa."
    if priority == "High":
        return "Trabajo bloqueado sin alternativa." if t["blocked"] and not t["workaround"] else "Plazo de hasta 24 horas e impacto significativo."
    if priority == "Medium":
        return "Existe alternativa de trabajo." if t["workaround"] else "Plazo de esta semana o impacto significativo sin urgencia mayor."
    return "Plazo lejano e impacto menor sin bloqueo ni incidente."

def expected_answer(e):
    return {"tasks": [{"id": t["id"], "priority": priority_for(t), "reason": reason_for(t)}
                      for t in e["tasks"]]}

save_json(run_dir / "policy.json", {"instruction": POLICY, "priorities": PRIORITIES,
          "task": "task_set_priority_classification", "scheduling": False})


## 5. Conjuntos de tareas sintéticos

Generamos tareas con descripción, plazo relativo en horas, impacto 1–5, bloqueo,
alternativa e incidente activo. Las etiquetas se derivan de la política, no se
asignan aleatoriamente. Equilibramos las prioridades globalmente por partición;
un conjunto puede repetir prioridades y no tiene que contener las cuatro.

Entrenamiento, validación y prueba usan servicios y redacciones diferentes. Se
guardan entradas, referencias, semilla y huella. Los datos son sintéticos y no
representan la distribución real de solicitudes de una empresa.


In [4]:
SERVICES = {
    "train": ["correo", "ventas", "inventario", "red interna", "nómina", "repositorio"],
    "validation": ["reservas", "intranet", "impresión", "almacén"],
    "test": ["facturación", "cursos", "control de acceso", "compras"],
}
ACTIONS = {"train": ["Revisar", "Resolver problema en", "Actualizar"],
           "validation": ["Verificar", "Atender solicitud de"],
           "test": ["Corregir", "Completar revisión de"]}

def make_task(rng, split, wanted):
    t = {"id": "", "task": f"{rng.choice(ACTIONS[split])} {rng.choice(SERVICES[split])}",
         "deadline_hours": rng.choice([48,72,240,336]), "impact": rng.randint(1,2),
         "blocked": False, "workaround": False, "active_incident": False}
    if wanted == "Critical":
        if rng.random() < .5:
            t["active_incident"] = True
        else:
            t.update(blocked=True,impact=5)
    elif wanted == "High":
        if rng.random() < .5:
            t.update(blocked=True,impact=rng.randint(1,4))
        else:
            t.update(deadline_hours=rng.choice([1,4,8,24]),impact=rng.randint(3,5))
    elif wanted == "Medium":
        t.update(deadline_hours=rng.choice([48,72,168]),impact=rng.randint(1,4))
        t["workaround"] = rng.random() < .5
    else:
        t["deadline_hours"] = rng.choice([240,336])
    if rng.random() < .25:
        t["task"] += "; solicitan atención URGENTE"
    assert priority_for(t) == wanted
    return t

def make_dataset():
    rng = random.Random(config.seed)
    rows=[]
    for split,size in {"train":config.train_size,"validation":config.validation_size,
                       "test":config.test_size}.items():
        # Equilibrio global por tarea; una lista puede repetir u omitir prioridades.
        pool=[make_task(rng,split,p) for p in PRIORITIES for _ in range(size)]
        rng.shuffle(pool)
        for i in range(size):
            tasks=pool[i*TASKS_PER_SET:(i+1)*TASKS_PER_SET]
            for j,t in enumerate(tasks,1):t["id"]=f"T{j}"
            text=json.dumps({"tasks":tasks},ensure_ascii=False,separators=(",",":"))
            rows.append({"id":hashlib.sha256(text.encode()).hexdigest()[:16],"split":split,
                "tasks":tasks,"input":text,
                "expected_priorities":{t["id"]:priority_for(t) for t in tasks},
                "provenance":"synthetic_task_priority_policy_v1"})
    return rows

examples=make_dataset()
assert len({e["input"] for e in examples})==len(examples)
splits={s:[e for e in examples if e["split"]==s] for s in ["train","validation","test"]}
for a,b in [("train","validation"),("train","test"),("validation","test")]:
    assert not ({e["id"] for e in splits[a]} & {e["id"] for e in splits[b]})
    assert not (set(SERVICES[a]) & set(SERVICES[b]))
assert any(len(set(e["expected_priorities"].values()))<4 for e in examples)
serialized="".join(json.dumps(e,ensure_ascii=False,sort_keys=True)+"\n" for e in examples)
(run_dir/"dataset.jsonl").write_text(serialized,encoding="utf-8")
dataset_hash=hashlib.sha256(serialized.encode()).hexdigest()
save_json(run_dir/"dataset_manifest.json",{"sha256":dataset_hash,"seed":config.seed,
    "task":"task_set_priority_classification","tasks_per_set":TASKS_PER_SET,
    "counts":{s:len(v) for s,v in splits.items()}})
flat=[{"partition":e["split"],"priority":priority_for(t)} for e in examples for t in e["tasks"]]
distribution=pd.crosstab(pd.DataFrame(flat).priority,pd.DataFrame(flat).partition).reindex(PRIORITIES)
distribution.to_csv(run_dir/"priority_distribution.csv")
display(distribution)
display(pd.DataFrame(splits["test"][0]["tasks"]))
ax=distribution[["train","validation","test"]].plot.bar(rot=0,color=["#326A9F","#A46E13","#98658B"])
ax.set(title="Tareas por prioridad y partición",ylabel="Tareas",xlabel="Prioridad")
ax.legend(loc="upper left",bbox_to_anchor=(1.02,1))
show_figure(ax.figure,"priority_distribution.png")


partition,test,train,validation
priority,,,
Critical,12,80,12
High,12,80,12
Medium,12,80,12
Low,12,80,12


,id,task,deadline_hours,impact,blocked,workaround,active_incident
0,T1,Corregir cursos,240,5,True,False,False
1,T2,Corregir compras; solicitan atención URGENTE,1,4,False,False,False
2,T3,Completar revisión de cursos; solicitan atenci...,72,3,False,False,False
3,T4,Completar revisión de cursos,48,4,False,True,False


## 6. Evaluar clasificación y cobertura del conjunto

Exactitud y macro-F1 se calculan sobre tareas individuales, alineando por ID.
Se informa también la fracción de conjuntos con todas las prioridades correctas,
completitud, duplicados, IDs desconocidos y cumplimiento JSON. Una tarea omitida
o duplicada se predice INVALID y cuenta como error; nunca se elimina del denominador.
La matriz conserva una columna INVALID. El orden de salida no afecta el resultado.

Una respuesta JSON/esquema inválida invalida el conjunto completo: sus cuatro
tareas cuentan como errores. Esta política mide resultados utilizables por una
aplicación y explica por qué errores de formato reducen la exactitud.


In [5]:
def parse_response(raw):
    def unique_object(pairs):
        obj={}
        for key,value in pairs:
            if key in obj:raise ValueError(f"Clave duplicada: {key}")
            obj[key]=value
        return obj
    try:
        obj=json.loads(raw,object_pairs_hook=unique_object,
                       parse_constant=lambda s:(_ for _ in ()).throw(ValueError(s)))
    except (ValueError,TypeError) as exc:
        return {"json_ok":False,"schema_ok":False,"items":[],"error":str(exc)}
    valid=isinstance(obj,dict) and set(obj)=={"tasks"} and isinstance(obj["tasks"],list)
    if valid:
        valid=all(isinstance(t,dict) and set(t)=={"id","priority","reason"}
            and isinstance(t["id"],str) and isinstance(t["priority"],str)
            and t["priority"] in PRIORITIES and isinstance(t["reason"],str)
            and bool(t["reason"].strip()) for t in obj["tasks"])
    return {"json_ok":True,"schema_ok":bool(valid),"items":obj["tasks"] if valid else [],
            "error":None if valid else "Se esperaba tasks con id, priority y reason válidos."}

def score_tasks(e,parsed):
    ids=[t["id"] for t in parsed["items"]]
    expected=e["expected_priorities"]
    valid=parsed["schema_ok"] and len(ids)==len(expected) and set(ids)==set(expected)
    scored=[]
    for task in e["tasks"]:
        matches=[t for t in parsed["items"] if t["id"]==task["id"]]
        predicted=matches[0]["priority"] if len(matches)==1 else "INVALID"
        scored.append({"task_id":task["id"],"task":task["task"],
            "expected":expected[task["id"]],"predicted":predicted,
            "reason":matches[0]["reason"] if len(matches)==1 else ""})
    return {"tasks_scored":scored,"valid_task_set":bool(valid),
        "completeness":len(set(ids)&set(expected))/len(expected),
        "duplicate_count":len(ids)-len(set(ids)),"unknown_count":len(set(ids)-set(expected)),
        "exact_set":bool(valid and all(t["expected"]==t["predicted"] for t in scored))}

def quality_metrics(rows):
    tasks=[t for row in rows for t in row["tasks_scored"]]
    actual=[t["expected"] for t in tasks];predicted=[t["predicted"] for t in tasks]
    recall=recall_score(actual,predicted,labels=PRIORITIES,average=None,zero_division=0)
    return {"n_sets":len(rows),"n_tasks":len(tasks),
        "accuracy":float(accuracy_score(actual,predicted)),
        "macro_f1":float(f1_score(actual,predicted,labels=PRIORITIES,average="macro",zero_division=0)),
        "exact_set_rate":float(np.mean([r["exact_set"] for r in rows])),
        "valid_task_set_rate":float(np.mean([r["valid_task_set"] for r in rows])),
        "completeness":float(np.mean([r["completeness"] for r in rows])),
        "json_rate":float(np.mean([r["json_ok"] for r in rows])),
        "schema_rate":float(np.mean([r["schema_ok"] for r in rows])),
        **{f"recall_{p}":float(v) for p,v in zip(PRIORITIES,recall)}}

e=splits["test"][0];perfect=parse_response(json.dumps(expected_answer(e)))
assert score_tasks(e,perfect)["exact_set"]
reordered={**perfect,"items":list(reversed(perfect["items"]))}
assert score_tasks(e,reordered)["exact_set"],"El orden de salida no debe cambiar la evaluación."
missing={**perfect,"items":perfect["items"][:-1]}
assert not score_tasks(e,missing)["valid_task_set"]
duplicate={**perfect,"items":[perfect["items"][0]]*TASKS_PER_SET}
assert score_tasks(e,duplicate)["duplicate_count"]==TASKS_PER_SET-1
bad=parse_response("no JSON");failed={**bad,**score_tasks(e,bad)}
assert quality_metrics([failed])["accuracy"]==0
assert quality_metrics([failed])["macro_f1"]==0
assert not parse_response('{"tasks":[{"id":"T1","priority":[],"reason":"x"}]}')["schema_ok"]
assert not parse_response('{"tasks":[{"id":"T1","priority":"Hich","reason":"x"}]}')["schema_ok"]
print("Verificados: cuatro prioridades, conjunto completo, duplicados y respuestas inválidas.")


Verificados: cuatro prioridades, conjunto completo, duplicados y respuestas inválidas.


## 7. Cargar Qwen y preparar el chat

Usamos la clase nativa `Qwen3_5ForConditionalGeneration`, entrada de texto y
`enable_thinking=False`. No reutilizamos los marcadores Gemma del ejemplo.
Las tres condiciones comparten NF4 de cuatro bits, doble cuantización, precisión
y generación greedy. Nunca hacemos offload automático a CPU para esconder falta de VRAM.

Los parámetros originales quedan congelados. El conjunto de módulos adaptados
se restringe a las proyecciones MLP del lenguaje, no a visión ni a convoluciones.

El preflight exige al menos 12 GiB libres antes de cargar pesos. Es una
comprobación preventiva, no una garantía de memoria ni una reserva de VRAM.


In [6]:
GPU_ENABLED = config.mode != "prepare"
if GPU_ENABLED:
    import torch
    import bitsandbytes as bnb
    from datasets import Dataset
    from transformers import (AutoTokenizer, BitsAndBytesConfig,
        Qwen3_5ForConditionalGeneration, Trainer, TrainerCallback, TrainingArguments, set_seed)
    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

    if not os.environ.get("SLURM_JOB_ID"):
        raise RuntimeError("En Khipu ejecutar dentro de una asignación Slurm GPU, nunca en login.")
    if not torch.cuda.is_available():
        raise RuntimeError("El job no tiene CUDA disponible; revisar GPU y driver.")
    set_seed(config.seed)
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    manifest = json.loads(Path("models/model_manifest.json").read_text())
    if manifest["model_id"] != config.model_id:
        raise ValueError("Checkpoint incorrecto en el manifiesto.")
    snapshot_path = Path(manifest["snapshot_path"])
    if not snapshot_path.is_dir():
        raise FileNotFoundError("Preparar la descarga antes del job; ruta local inexistente.")
    tokenizer = AutoTokenizer.from_pretrained(snapshot_path, local_files_only=True)
    declared_tokens = json.loads((snapshot_path / "tokenizer_config.json").read_text())["added_tokens_decoder"]
    expected_eos = next(int(i) for i, v in declared_tokens.items()
                        if v["content"] == tokenizer.eos_token)
    assert tokenizer.eos_token_id == expected_eos, "Tokenizer incompleto o incompatible."
    assert len(tokenizer) > 200_000 and len(tokenizer.encode("Necesito revisar el sistema")) > 2
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    environment.update({"cuda_free_gib_at_start": free_bytes / 2**30,
                        "slurm_node": os.environ.get("SLURMD_NODENAME"),
                        "slurm_job_gpus": os.environ.get("SLURM_JOB_GPUS"),
                        "slurm_tres_per_node": os.environ.get("SLURM_TRES_PER_NODE"),
                        "gpu": torch.cuda.get_device_name(0),
                        "gpu_total_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
                        "torch_cuda": torch.version.cuda, "dtype": str(compute_dtype),
                        "model": manifest, "dataset_sha256": dataset_hash, "status": "running"})
    save_json(run_dir / "environment.json", environment)
    print(environment["gpu"], compute_dtype, "commit", manifest["revision"])
    if free_bytes < 12 * 2**30:
        raise RuntimeError("Menos de 12 GiB VRAM libres. Solicitar una GPU dedicada; no se cargaron pesos.")

def render_prompt(text):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": POLICY}, {"role": "user", "content": text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)

def load_original(training=False):
    set_seed(config.seed)
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        snapshot_path, local_files_only=True, quantization_config=quantization,
        dtype=compute_dtype, device_map={"": 0}, attn_implementation="sdpa")
    model.requires_grad_(False)
    if training:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        model.config.use_cache = False
        model.config.text_config.use_cache = False
    else:
        model.eval()
    return model

def release_memory():
    gc.collect()
    torch.cuda.empty_cache()

if GPU_ENABLED:
    example_prompt = render_prompt(splits["train"][0]["input"])
    assert "<start_of_turn>" not in example_prompt
    assert example_prompt.endswith("<think>\n\n</think>\n\n"), (
        "El template cambió: comprobar desactivación de thinking antes de continuar.")
    display(Markdown("**Prompt real de ejemplo:**"))
    print(example_prompt)


/home/jeffrey.monja/Github/2026-II/PlanIA/S10/Lora Scripts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NVIDIA RTX A6000 torch.bfloat16 commit 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


**Prompt real de ejemplo:**

<|im_start|>system
Asigna una prioridad a CADA tarea del conjunto. Usa exactamente Critical, High, Medium o Low. Aplica la primera regla satisfecha: Critical si active_incident=true, o si blocked=true, impact=5 y workaround=false. High si blocked=true y workaround=false, o si deadline_hours<=24 e impact>=3. Medium si workaround=true, deadline_hours<=168 o impact>=3. Low en los demás casos. impact va de 1 (menor) a 5 (crítico). La insistencia de la descripción no cambia los criterios explícitos. Incluye cada id exactamente una vez, con una razón breve de hasta 12 palabras. Devuelve solo JSON: {"tasks":[{"id":"T1","priority":"High","reason":"..."},...]}. Conserva los IDs recibidos.<|im_end|>
<|im_start|>user
{"tasks":[{"id":"T1","task":"Resolver problema en ventas","deadline_hours":72,"impact":1,"blocked":false,"workaround":false,"active_incident":false},{"id":"T2","task":"Revisar repositorio; solicitan atención URGENTE","deadline_hours":48,"impact":3,"blocked":false,"workaround":false,"

## 8. Entrenar sobre respuestas estructuradas

La respuesta esperada incluye ID, prioridad y razón para cada tarea. Solo sus
tokens contribuyen a la pérdida; prompt y padding usan -100. El collator conserva
las etiquetas. No se truncan silenciosamente respuestas que excedan el límite.


In [7]:
def tokenize_row(example):
    prefix = tokenizer.encode(render_prompt(example["input"]), add_special_tokens=False)
    answer = json.dumps(expected_answer(example),
                        ensure_ascii=False, separators=(",", ":"))
    suffix = tokenizer.encode(answer, add_special_tokens=False) + [tokenizer.eos_token_id]
    ids = prefix + suffix
    if len(ids) > config.max_seq_length:
        raise ValueError(f"Ejemplo {example['id']}: {len(ids)} tokens supera el límite; no se truncó.")
    return {"input_ids": ids, "attention_mask": [1] * len(ids),
            "labels": [-100] * len(prefix) + suffix}

def answer_collator(features):
    width = max(len(f["input_ids"]) for f in features)
    ids, masks, labels = [], [], []
    for f in features:
        pad = width - len(f["input_ids"])
        ids.append(f["input_ids"] + [tokenizer.pad_token_id] * pad)
        masks.append(f["attention_mask"] + [0] * pad)
        labels.append(f["labels"] + [-100] * pad)
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(masks),
            "labels": torch.tensor(labels)}

if GPU_ENABLED:
    tokenized = {s: [tokenize_row(e) for e in rows] for s, rows in splits.items()}
    for s, rows in tokenized.items():
        assert all(any(x != -100 for x in r["labels"]) for r in rows)
        print(s, "longitud máxima:", max(len(r["input_ids"]) for r in rows))
    train_dataset = Dataset.from_list(tokenized["train"])
    validation_dataset = Dataset.from_list(tokenized["validation"])
    pair = answer_collator(tokenized["train"][:2])
    assert torch.all(pair["labels"][pair["attention_mask"] == 0] == -100)
    first = tokenized["train"][0]
    answer_start = next(i for i, label in enumerate(first["labels"]) if label != -100)
    assert all(label == -100 for label in first["labels"][:answer_start])
    assert tokenizer.decode(first["labels"][answer_start:-1]).startswith('{"tasks":')


train longitud máxima: 460
validation longitud máxima: 446
test longitud máxima: 468


## 9. Evaluación común y recursos

La latencia mide tokenización, generación y decodificación después de calentar el
modelo; sincronizamos CUDA antes/después. No incluye carga del checkpoint.
Los picos CUDA son memoria **asignada y reservada por este proceso**, no uso total
del dispositivo. Medimos entrenamiento e inferencia por separado.
Los adaptadores se evalúan recargados y sin fusionar. No usamos cachés especiales de DoRA.

Si se utiliza una GPU compartida, otros jobs pueden variar su carga durante
la comparación. Los tiempos son observaciones de esta ejecución, no una prueba
controlada de velocidad relativa entre métodos.


In [8]:
all_predictions, all_metrics, training_histories = [], [], {}

def generate_one(model, text):
    torch.cuda.synchronize()
    started = time.perf_counter()
    inputs = tokenizer(render_prompt(text), return_tensors="pt", add_special_tokens=False).to("cuda:0")
    stop_ids = sorted({tokenizer.eos_token_id,
                       tokenizer.convert_tokens_to_ids("<|endoftext|>")})
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=config.max_new_tokens,
            do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id,
            eos_token_id=stop_ids)
    new_tokens = output[0, inputs.input_ids.shape[1]:]
    # Remove only the terminal stop token; keep unexpected special markers visible.
    visible_ids = new_tokens.tolist()
    if visible_ids and visible_ids[-1] in stop_ids:
        visible_ids = visible_ids[:-1]
    raw = tokenizer.decode(visible_ids, skip_special_tokens=False).strip()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    return raw, elapsed, len(new_tokens)

def persist_results():
    if all_predictions:
        responses=[{k:v for k,v in r.items() if k!="tasks_scored"} for r in all_predictions]
        pd.DataFrame(responses).to_csv(run_dir / "responses.csv",index=False)
        tasks=[{"method":r["method"],"example_id":r["example_id"],**t}
               for r in all_predictions for t in r["tasks_scored"]]
        pd.DataFrame(tasks).to_csv(run_dir / "predictions.csv",index=False)
    if all_metrics:
        pd.DataFrame(all_metrics).to_csv(run_dir / "metrics.csv", index=False)
        save_json(run_dir / "metrics.json", all_metrics)

def evaluate(model, method, training_info=None):
    model.eval()
    model.config.use_cache = True
    model.config.text_config.use_cache = True
    generate_one(model, splits["train"][0]["input"])
    torch.cuda.reset_peak_memory_stats()
    rows = []
    for i, example in enumerate(splits["test"], 1):
        raw, latency, generated_tokens = generate_one(model, example["input"])
        rows.append({"method": method, "example_id": example["id"],
            "expected_priorities_json": json.dumps(example["expected_priorities"]), "input": example["input"],
            "raw_output": raw, **{k: v for k, v in parse_response(raw).items() if k != "items"},
            **score_tasks(example, parse_response(raw)),
            "latency_seconds": latency, "generated_tokens": generated_tokens})
        if i % 10 == 0 or i == len(splits["test"]):
            print(method, i, "/", len(splits["test"]))
    latencies = [r["latency_seconds"] for r in rows]
    metrics = {"method": method, **quality_metrics(rows),
        "mean_generated_tokens": float(np.mean([r["generated_tokens"] for r in rows])),
        "mean_latency_seconds": float(np.mean(latencies)),
        "median_latency_seconds": float(np.median(latencies)),
        "p95_latency_seconds": float(np.percentile(latencies, 95)),
        "inference_peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
        "inference_peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
        "train_seconds": None, "train_peak_allocated_gib": None,
        "train_peak_reserved_gib": None, "trainable_parameters": 0,
        **(training_info or {})}
    all_predictions.extend(rows)
    all_metrics.append(metrics)
    persist_results()
    display(pd.DataFrame([metrics]))
    return metrics


## 10. Qwen original: priorizar antes de adaptar

Clasificar el test completo antes del entrenamiento. El mismo test se conserva
para LoRA y DoRA; no selecciona hiperparámetros ni checkpoints.


In [9]:
if GPU_ENABLED:
    original = load_original()
    evaluate(original, "Original")
    del original
    release_memory()
else:
    print("prepare: listas y evaluación verificadas; no se ejecutó el LLM.")


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<02:10,  5.54it/s]

Loading weights:   1%|          | 6/723 [00:00<00:29, 24.48it/s]

Loading weights:   2%|▏         | 14/723 [00:00<00:15, 45.38it/s]

Loading weights:   4%|▎         | 27/723 [00:00<00:09, 71.62it/s]

...

[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Original 10 / 12


Original 12 / 12


,method,n_sets,n_tasks,accuracy,macro_f1,exact_set_rate,valid_task_set_rate,completeness,json_rate,schema_rate,...,mean_generated_tokens,mean_latency_seconds,median_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_seconds,train_peak_allocated_gib,train_peak_reserved_gib,trainable_parameters
0,Original,12,48,0.416667,0.551901,0.25,0.5,0.5,0.5,0.5,...,91.333333,5.569262,5.273603,7.814076,3.240171,3.261719,None,None,None,0


## 11. Entrenamiento compartido de LoRA y DoRA

Ambos parten de cargas independientes del Qwen original, con NF4 4-bit, r=16,
alpha=32, dropout=0, lr=2e-4 y la misma partición. Deadline usa una época de 80
conjuntos; full usa tres. La selección usa menor pérdida de validación, sin test.
DoRA utiliza `use_dora=True` de PEFT. Verificamos gradientes finitos/no nulos,
actualización de parámetros, guardado y recarga.

Transformers 5.19 acepta 0.03 como fracción en warmup_steps. Las adaptaciones
actúan sobre 96 proyecciones MLP del lenguaje; visión permanece congelada.


In [10]:
if GPU_ENABLED:
    class GradientAudit(TrainerCallback):
        def __init__(self):
            self.checked = False

        def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
            grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
            if not grads or not all(torch.isfinite(g).all().item() for g in grads):
                raise RuntimeError("Gradientes ausentes o no finitos.")
            if not self.checked and not any(torch.count_nonzero(g).item() for g in grads):
                raise RuntimeError("Todos los gradientes son cero antes de la primera actualización.")
            self.checked = True

def target_modules_for(model):
    targets = [name for name, layer in model.named_modules()
        if "language_model" in name.split(".") and ".mlp." in name
        and name.rsplit(".", 1)[-1] in {"gate_proj", "up_proj", "down_proj"}
        and isinstance(layer, (torch.nn.Linear, bnb.nn.Linear4bit))]
    if len(targets) != model.config.text_config.num_hidden_layers * 3:
        raise RuntimeError(f"Se esperaban tres proyecciones MLP por capa: {len(targets)} encontradas.")
    assert all("visual" not in name and "vision" not in name for name in targets)
    return targets

def train_adapter(method):
    release_memory()
    base = load_original(training=True)
    targets = target_modules_for(base)
    save_json(run_dir / f"{method.lower()}_targets.json", targets)
    adapter_config = LoraConfig(r=config.rank, lora_alpha=config.alpha,
        lora_dropout=config.dropout, target_modules=targets, bias="none",
        task_type="CAUSAL_LM", use_dora=(method == "DoRA"))
    model = get_peft_model(base, adapter_config)
    trainable = {name: p for name, p in model.named_parameters() if p.requires_grad}
    assert trainable and all("lora_" in name for name in trainable)
    # Una pequeña matriz B permite comprobar la actualización sin copiar el modelo.
    probe_name = next(n for n in trainable if "lora_B" in n)
    probe_before = trainable[probe_name].detach().cpu().clone()
    trainable_count = sum(p.numel() for p in trainable.values())
    model.print_trainable_parameters()
    smoke = config.mode == "smoke"
    args = TrainingArguments(
        output_dir=str(run_dir / "checkpoints" / method.lower()),
        num_train_epochs=config.epochs, max_steps=2 if smoke else -1,
        per_device_train_batch_size=config.batch_size, per_device_eval_batch_size=1,
        gradient_accumulation_steps=config.accumulation, learning_rate=config.learning_rate,
        weight_decay=0.01, warmup_steps=config.warmup_ratio, lr_scheduler_type="cosine",
        eval_strategy="steps" if smoke else "epoch", eval_steps=2 if smoke else None,
        save_strategy="steps" if smoke else "epoch", save_steps=2 if smoke else 500,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        save_total_limit=1, logging_steps=1 if smoke else 5, report_to="none",
        bf16=(compute_dtype == torch.bfloat16), fp16=(compute_dtype == torch.float16),
        optim="paged_adamw_8bit", gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        seed=config.seed, data_seed=config.seed, dataloader_num_workers=0,
        eval_accumulation_steps=1, prediction_loss_only=True)
    audit = GradientAudit()
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset,
        eval_dataset=validation_dataset, data_collator=answer_collator, callbacks=[audit])
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    result = trainer.train()
    torch.cuda.synchronize()
    info = {"train_seconds": time.perf_counter() - started,
            "train_peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
            "train_peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
            "trainable_parameters": trainable_count}
    assert math.isfinite(result.training_loss) and audit.checked
    assert not torch.equal(probe_before, trainable[probe_name].detach().cpu()), "No hubo actualización."
    adapter_path = run_dir / "adapters" / method.lower()
    trainer.model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    history = pd.DataFrame(trainer.state.log_history)
    history.to_csv(run_dir / f"{method.lower()}_training_history.csv", index=False)
    training_histories[method] = history
    save_json(run_dir / f"{method.lower()}_training.json", {
        **info, "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_loss": float(trainer.state.best_metric),
        "finite_nonzero_gradients": audit.checked, "parameter_updated": True,
        "optimizer_steps": trainer.state.global_step})
    del trainer, trainable, model, base
    release_memory()
    # Prueba de guardado/recarga y evaluación con un checkpoint base fresco.
    restored_base = load_original()
    restored = PeftModel.from_pretrained(restored_base, adapter_path, is_trainable=False)
    assert restored.peft_config["default"].use_dora == (method == "DoRA")
    evaluate(restored, method, info)
    del restored, restored_base
    release_memory()


## 12. LoRA

Entrenar, guardar y recargar el adaptador sobre un Qwen original nuevo. Evaluar
las cuatro prioridades y conservar respuestas completas, razones y predicciones.


In [11]:
if GPU_ENABLED:
    train_adapter("LoRA")


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<01:42,  7.06it/s]

Loading weights:   2%|▏         | 13/723 [00:00<00:11, 61.63it/s]

Loading weights:   3%|▎         | 24/723 [00:00<00:08, 80.65it/s]

Loading weights:   5%|▍         | 35/723 [00:00<00:07, 88.69it/s]

...

Loading weights:  98%|█████████▊| 710/723 [00:05<00:00, 165.61it/s]

Loading weights: 100%|██████████| 723/723 [00:05<00:00, 121.55it/s]

LoRA 10 / 12


LoRA 12 / 12


,method,n_sets,n_tasks,accuracy,macro_f1,exact_set_rate,valid_task_set_rate,completeness,json_rate,schema_rate,...,mean_generated_tokens,mean_latency_seconds,median_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_seconds,train_peak_allocated_gib,train_peak_reserved_gib,trainable_parameters
0,LoRA,12,48,0.854167,0.857163,0.583333,1.0,1.0,1.0,1.0,...,85.583333,6.172143,6.117962,6.998536,5.690665,5.792969,197.977814,7.026218,8.923828,18087936


## 13. DoRA con PEFT

Repetir desde el mismo Qwen original. DoRA aprende magnitud y dirección; se conserva
el mismo presupuesto y la misma política. Los adaptadores se evalúan sin fusionar
y sin DoraCaching; el tiempo observado depende de esa implementación.


In [12]:
if GPU_ENABLED:
    train_adapter("DoRA")


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<01:42,  7.04it/s]

Loading weights:   2%|▏         | 13/723 [00:00<00:11, 59.95it/s]

Loading weights:   4%|▎         | 26/723 [00:00<00:08, 84.65it/s]

Loading weights:   5%|▌         | 37/723 [00:00<00:07, 92.67it/s]

...

Loading weights:  98%|█████████▊| 707/723 [00:05<00:00, 160.00it/s]

Loading weights: 100%|██████████| 723/723 [00:05<00:00, 122.48it/s]

DoRA 10 / 12


DoRA 12 / 12


,method,n_sets,n_tasks,accuracy,macro_f1,exact_set_rate,valid_task_set_rate,completeness,json_rate,schema_rate,...,mean_generated_tokens,mean_latency_seconds,median_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_seconds,train_peak_allocated_gib,train_peak_reserved_gib,trainable_parameters
0,DoRA,12,48,0.833333,0.832751,0.583333,1.0,1.0,1.0,1.0,...,86.166667,49.622163,49.016465,66.816341,8.415735,8.873047,367.184325,9.408908,10.912109,18759680


## 14. Comparar las prioridades asignadas

Figuras de exactitud por tarea, macro-F1, conjuntos correctos y cobertura. Las
matrices muestran qué prioridades se confunden y mantienen los resultados inválidos.
Una ejecución por método no permite afirmar significancia ni robustez entre semillas.


In [13]:
if GPU_ENABLED:
    metrics_df=pd.DataFrame(all_metrics).set_index("method").loc[list(COLORS)]
    display(metrics_df[["n_sets","n_tasks","accuracy","macro_f1","exact_set_rate",
                        "valid_task_set_rate","json_rate","schema_rate"]])
    fig,ax=plt.subplots(figsize=(9,4));x=np.arange(4)
    fields=["accuracy","macro_f1","exact_set_rate","valid_task_set_rate"]
    for i,method in enumerate(COLORS):
        bars=ax.bar(x+(i-1)*.24,metrics_df.loc[method,fields].to_numpy(dtype=float)*100,.24,
                    label=method,color=COLORS[method])
        ax.bar_label(bars,fmt="%.1f",fontsize=8)
    ax.set(xticks=x,xticklabels=["Exactitud/tarea","Macro-F1","Conjunto correcto","Conjunto completo"],
           ylim=(0,112),ylabel="Porcentaje (%)",
           title=f"Prioridades de {len(splits['test'])*TASKS_PER_SET} tareas ({config.mode})")
    ax.legend(loc="upper left",bbox_to_anchor=(1.02,1))
    show_figure(fig,"priority_comparison.png")
    fig,axes=plt.subplots(1,3,figsize=(15,4),sharey=True)
    columns=PRIORITIES+["INVALID"];matrices={}
    for method in COLORS:
        tasks=[t for r in all_predictions if r["method"]==method for t in r["tasks_scored"]]
        matrix=np.array([[sum(t["expected"]==p and t["predicted"]==q for t in tasks)
                          for q in columns] for p in PRIORITIES])
        assert matrix.sum()==len(splits["test"])*TASKS_PER_SET
        matrices[method]=matrix
        pd.DataFrame(matrix,index=PRIORITIES,columns=columns).to_csv(
            run_dir/f"{method.lower()}_confusion.csv",index_label="expected")
    maximum=max(int(m.max()) for m in matrices.values())
    for ax,(method,matrix) in zip(axes,matrices.items()):
        ax.imshow(matrix,cmap="Blues",vmin=0,vmax=max(1,maximum))
        for (i,j),value in np.ndenumerate(matrix):
            ax.text(j,i,str(value),ha="center",va="center",
                    color="white" if value>maximum/2 else "black")
        ax.set(title=method,xticks=range(5),xticklabels=columns,
               yticks=range(4),yticklabels=PRIORITIES,xlabel="Prioridad predicha")
        ax.tick_params(axis="x",rotation=30)
    axes[0].set_ylabel("Prioridad de referencia")
    show_figure(fig,"priority_confusion.png")


,n_sets,n_tasks,accuracy,macro_f1,exact_set_rate,valid_task_set_rate,json_rate,schema_rate
method,,,,,,,,
Original,12,48,0.416667,0.551901,0.250000,0.5,0.5,0.5
LoRA,12,48,0.854167,0.857163,0.583333,1.0,1.0,1.0
DoRA,12,48,0.833333,0.832751,0.583333,1.0,1.0,1.0


## 15. Pérdida, tiempo y memoria

Las curvas muestran pérdida de entrenamiento y validación, no una tasa de acierto.
El original no tiene entrenamiento: ese tiempo es ausente, no una medición de cero.
Los picos son asignaciones del proceso durante etapas secuenciales; pueden retener
asignaciones previas y no representan requisitos aislados de cada adaptador.
La GPU es compartida: los tiempos no son una prueba controlada de velocidad.

Si solo se registra un valor de entrenamiento y uno de validación, mostramos
barras con esos valores, sin dibujar una evolución temporal inexistente.


In [2]:
if GPU_ENABLED:
    fig, ax = plt.subplots()
    enough_points = any("loss" in h and h["loss"].notna().sum() > 1
                        for h in training_histories.values())
    if enough_points:
        for method, history in training_histories.items():
            for field, style, label in [("loss", "-", "entrenamiento"),
                                        ("eval_loss", "--", "validación")]:
                if field in history:
                    values = history.dropna(subset=[field])
                    ax.plot(values["step"], values[field], style, marker="o",
                            color=COLORS[method], label=f"{method}: {label}")
        ax.set(xlabel="Paso del optimizador", ylabel="Pérdida (cross-entropy)",
               title="Pérdidas registradas durante el entrenamiento")
    else:
        x = np.arange(2)
        for i, (method, history) in enumerate(training_histories.items()):
            values = [float(history["loss"].dropna().iloc[-1]),
                      float(history["eval_loss"].dropna().iloc[-1])]
            bars = ax.bar(x + (i-.5)*.32, values, .32, label=method, color=COLORS[method])
            ax.bar_label(bars, fmt="%.3f", padding=3)
        ax.set(xticks=x, xticklabels=["Entrenamiento\n(promedio registrado)", "Validación\n(final)"],
               ylabel="Pérdida (cross-entropy)", title="Pérdidas registradas al finalizar la época")
        ax.margins(y=.2)
    ax.legend()
    show_figure(fig, "loss_curves.png")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    views = [("mean_latency_seconds", "Latencia media de inferencia", "Segundos / conjunto", list(COLORS)),
             ("train_seconds", "Duración del entrenamiento", "Segundos", ["LoRA", "DoRA"]),
             ("train_peak_allocated_gib", "Pico asignado (entrenamiento)", "GiB", ["LoRA", "DoRA"])]
    for ax, (field, title, unit, methods) in zip(axes, views):
        bars = ax.bar(methods, [metrics_df.loc[m, field] for m in methods],
                      color=[COLORS[m] for m in methods])
        ax.bar_label(bars, fmt="%.2f", fontsize=9, padding=3)
        ax.set(title=title, ylabel=unit)
        ax.margins(y=.18)
    show_figure(fig, "resource_comparison.png")
    display(metrics_df[["trainable_parameters", "train_seconds", "mean_latency_seconds",
                        "p95_latency_seconds", "inference_peak_allocated_gib",
                        "inference_peak_reserved_gib", "train_peak_allocated_gib",
                        "train_peak_reserved_gib"]])


,trainable_parameters,train_seconds,mean_latency_seconds,p95_latency_seconds,inference_peak_allocated_gib,inference_peak_reserved_gib,train_peak_allocated_gib,train_peak_reserved_gib
method,,,,,,,,
Original,0,NaN,5.569262,7.814076,3.240171,3.261719,NaN,NaN
LoRA,18087936,197.977814,6.172143,6.998536,5.690665,5.792969,7.026218,8.923828
DoRA,18759680,367.184325,49.622163,66.816341,8.415735,8.873047,9.408908,10.912109


## 16. Ejemplo por tarea y conclusiones

Mostrar cada tarea con su prioridad de referencia, la predicha y su razón, por
modelo. Las conclusiones se calculan desde los resultados medidos. Un empate o
degradación también es un resultado válido.


In [3]:
if GPU_ENABLED:
    display(Markdown("### Ejemplo: prioridades y razones para un conjunto de tareas"))
    demo=splits["test"][0]
    display(pd.DataFrame(demo["tasks"]))
    for method in COLORS:
        response=next(r for r in all_predictions if r["method"]==method and r["example_id"]==demo["id"])
        display(Markdown(f"**{method}**"))
        display(pd.DataFrame(response["tasks_scored"])[["task_id","task","expected","predicted","reason"]])
    lines=["### Resultados observados"]
    for method in COLORS:
        row=metrics_df.loc[method]
        lines.append(f"- {method}: exactitud por tarea {row.accuracy:.1%}, macro-F1 {row.macro_f1:.3f}, "
                     f"conjuntos completos y sin errores de prioridad {row.exact_set_rate:.1%}.")
    for method in ["LoRA","DoRA"]:
        delta=metrics_df.loc[method,"macro_f1"]-metrics_df.loc["Original","macro_f1"]
        lines.append(f"- Cambio macro-F1 de {method} frente al original: {delta:+.3f}.")
    lines.append("Las razones se conservan para revisión cualitativa; su presencia no prueba su corrección.")
    lines.append("Datos sintéticos, una semilla y GPU compartida limitan las conclusiones y los tiempos. "
                 "Los picos de memoria del proceso secuencial pueden incluir asignaciones retenidas.")
    if config.mode=="smoke":lines.append("**Smoke de integración; no usar como evaluación final.**")
    conclusion="\n\n".join(lines);display(Markdown(conclusion))
    (run_dir/"conclusions.md").write_text(conclusion+"\n",encoding="utf-8")
    persisted=pd.read_csv(run_dir/"predictions.csv",keep_default_na=False)
    errors = persisted[persisted.expected != persisted.predicted]
    if len(errors):
        display(Markdown("### Ejemplos de errores conservados para revisión"))
        display(errors[["method","task_id","task","expected","predicted","reason"]].head(8))
    assert len(persisted)==len(splits["test"])*TASKS_PER_SET*3
    assert not persisted.duplicated(["method","example_id","task_id"]).any()
    for method in COLORS:
        rows=persisted[persisted.method==method]
        assert np.isclose(accuracy_score(rows.expected,rows.predicted),metrics_df.loc[method,"accuracy"])
        assert np.isclose(f1_score(rows.expected,rows.predicted,labels=PRIORITIES,average="macro",zero_division=0),
                          metrics_df.loc[method,"macro_f1"])
    environment.update(status="completed",task="task_set_priority_classification")
    save_json(run_dir/"environment.json",environment)
else:
    environment.update(status="data_checks_completed_no_llm_execution",task="task_set_priority_classification")
    save_json(run_dir/"environment.json",environment)
print("Artefactos:",run_dir.resolve())


### Ejemplo: prioridades y razones para un conjunto de tareas

,active_incident,blocked,deadline_hours,id,impact,task,workaround
0,False,True,240,T1,5,Corregir cursos,False
1,False,False,1,T2,4,Corregir compras; solicitan atención URGENTE,False
2,False,False,72,T3,3,Completar revisión de cursos; solicitan atenci...,False
3,False,False,48,T4,4,Completar revisión de cursos,True


**Original**

,task_id,task,expected,predicted,reason
0,T1,Corregir cursos,Critical,INVALID,
1,T2,Corregir compras; solicitan atención URGENTE,High,INVALID,
2,T3,Completar revisión de cursos; solicitan atenci...,Medium,INVALID,
3,T4,Completar revisión de cursos,Medium,INVALID,


**LoRA**

...

,task_id,task,expected,predicted,reason
0,T1,Corregir cursos,Critical,High,Tarea bloqueada sin alternativa.
1,T2,Corregir compras; solicitan atención URGENTE,High,High,Plazo de hasta 24 horas.
2,T3,Completar revisión de cursos; solicitan atenci...,Medium,Medium,Plazo de hasta 168 horas.
3,T4,Completar revisión de cursos,Medium,Medium,Existe alternativa.


### Resultados observados

- Original: exactitud por tarea 41.7%, macro-F1 0.552, conjuntos completos y sin errores de prioridad 25.0%.

- LoRA: exactitud por tarea 85.4%, macro-F1 0.857, conjuntos completos y sin errores de prioridad 58.3%.

- DoRA: exactitud por tarea 83.3%, macro-F1 0.833, conjuntos completos y sin errores de prioridad 58.3%.

- Cambio macro-F1 de LoRA frente al original: +0.305.

- Cambio macro-F1 de DoRA frente al original: +0.281.

Las razones se conservan para revisión cualitativa; su presencia no prueba su corrección.

Datos sintéticos, una semilla y GPU compartida limitan las conclusiones y los tiempos. Los picos de memoria del proceso secuencial pueden incluir asignaciones retenidas.

### Ejemplos de errores conservados para revisión

,method,task_id,task,expected,predicted,reason
0,Original,T1,Corregir cursos,Critical,INVALID,
1,Original,T2,Corregir compras; solicitan atención URGENTE,High,INVALID,
2,Original,T3,Completar revisión de cursos; solicitan atenci...,Medium,INVALID,
3,Original,T4,Completar revisión de cursos,Medium,INVALID,
4,Original,T1,Corregir cursos,Low,INVALID,
5,Original,T2,Corregir control de acceso,Low,INVALID,
6,Original,T3,Corregir compras,Critical,INVALID,
7,Original,T4,Completar revisión de cursos,Critical,INVALID,


Artefactos: /home/jeffrey.monja/Github/2026-II/PlanIA/S10/Lora Scripts/outputs/58321_deadline


## Referencias y siguientes pasos

- [Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B)
- [DoRA en PEFT](https://huggingface.co/docs/peft/package_reference/lora_variant_dora)
- [Khipu](https://docs.khipu.utec.edu.pe/guia-de-usuario/enviar-jobs/)

La tarea confirmada es asignar Critical/High/Medium/Low a cada tarea de un conjunto.
Para profundizar: datos reales, revisión humana de razones, otras políticas y
réplicas. Construir horarios requeriría duración y disponibilidad, y queda fuera.
